In [1]:
%cd /home/smalani/PartialObservations_BF/PartialObservations

import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
from URPModel import URP_metrics
from URPModel.datagen import f_cstr, get_pars
from numpy.random import default_rng
import torch
from scipy.integrate import solve_ivp
from sklearn.preprocessing import MinMaxScaler

/home/smalani/PartialObservations_BF/PartialObservations


In [2]:
from config import config
config["MODEL"]["BOX"] = 'Grey'
config["MODEL"]["Parameters"] = 'Fixed'

filename = '/home/smalani/PartialObservations_BF/PartialObservations/data/Case8a/' + "model_run_" + "2" + ".net"
network = URP_metrics.load_network(filename)

/home/smalani/PartialObservations_BF/PartialObservations/data/Case8a/model_run_2.net


In [3]:
def my_ode(t, x, Da, B, beta):
    x_input = torch.from_numpy(x).to(network.device)
    if x_input.ndim > 1:
        par_input = (torch.zeros((x_input.shape[1],1)) + Da).to(network.device)
    else:
        par_input = (torch.zeros((1,)) + Da).to(network.device)
    # par_input = torch.tensor([Da]).to(network.device)
    # while par_input.ndim < x_input.ndim:
    #     par_input = par_input.unsqueeze(0)
    g = network.raw_output(x_input.T, par_input).detach().cpu().numpy()

    x1, x2 = x
    dx1dt = -x1 + g
    dx2dt = -x2 + B * g - beta * x2
    return [dx1dt, dx2dt]

In [5]:

unstable_ss, unstable_Da, stable_ss1, stable_Da1, stable_ss2, stable_Da2, Da_arr, x1min, x2min, x1max, x2max, x1LC, x2LC = \
        URP_metrics.make_Bifurc_true_with_LC()


T = 1
teval = np.linspace(0, T, 100)

B = 11
beta = 3

B_arr = np.sort(
            np.concatenate((
                np.linspace(10,16,100),
                np.array([11])
            ))
        )

L2_norm_arr_B = np.zeros(B_arr.size)

for j in range(B_arr.size):
    B = B_arr[j]

    Da_sampler = default_rng(seed=123)
    LC_sampler = default_rng(seed=456)

    Da_sample = np.zeros(100)
    x1_sample = np.zeros(100)
    x2_sample = np.zeros(100)


    RHS_hair_error_trueLC_True = []
    RHS_hair_error_trueLC_Pred = []

    for i in range(x1_sample.size):
        Da_index = Da_sampler.choice(len(Da_arr))
        LC_index = LC_sampler.choice(np.array(x1LC[Da_index]).size)

        init = np.array([np.array(x1LC[Da_index]).reshape((-1))[LC_index],\
                            np.array(x2LC[Da_index]).reshape((-1))[LC_index]])
        pars_true = get_pars(Da_arr[Da_index], beta=beta, B=B)
        pars_ANN = np.array([Da_arr[Da_index]]).reshape((-1))

        sol_true = solve_ivp(f_cstr, y0=init, t_span=[0, T],
                                t_eval=teval, args=pars_true,
                                rtol=1e-5, atol=1e-8)

        sol_pred = solve_ivp(my_ode, y0=init, t_span=[0, T],
                                t_eval=teval, args=(pars_ANN, B, beta),
                                rtol=1e-5, atol=1e-8)

        RHS_hair_error_trueLC_True.append(sol_true.y[:,-1])
        RHS_hair_error_trueLC_Pred.append(sol_pred.y[:,-1])

    RHS_hair_error_trueLC_Pred = np.array(RHS_hair_error_trueLC_Pred)
    RHS_hair_error_trueLC_True = np.array(RHS_hair_error_trueLC_True)

    scaler = MinMaxScaler()
    scaler.fit(RHS_hair_error_trueLC_True)

    RHS_hair_error_trueLC_Pred_norm = scaler.transform(RHS_hair_error_trueLC_Pred)
    RHS_hair_error_trueLC_True_norm = scaler.transform(RHS_hair_error_trueLC_True)

    L2_norm = np.sum(np.sqrt(np.sum(((RHS_hair_error_trueLC_Pred_norm - RHS_hair_error_trueLC_True_norm)**2), axis=1)), axis=0) / RHS_hair_error_trueLC_True_norm.size

    L2_norm_arr_B[j] = L2_norm

    if j % 10 == 0:
        print(j, "B = ", B)

0 B =  10.0
1 B =  10.06060606060606
2 B =  10.121212121212121
3 B =  10.181818181818182
4 B =  10.242424242424242
5 B =  10.303030303030303
6 B =  10.363636363636363
7 B =  10.424242424242424
8 B =  10.484848484848484
9 B =  10.545454545454545
10 B =  10.606060606060606
11 B =  10.666666666666666
12 B =  10.727272727272727
13 B =  10.787878787878787
14 B =  10.848484848484848
15 B =  10.90909090909091
16 B =  10.969696969696969
17 B =  11.0


/home/smalani/PartialObservations_BF/PartialObservations/URPModel/datagen.py:24: RuntimeWarning: overflow encountered in exp
  dx1dt = -x1 + Da * (1-x1) * np.exp(x2)
/home/smalani/PartialObservations_BF/PartialObservations/URPModel/datagen.py:25: RuntimeWarning: overflow encountered in exp
  dx2dt = -x2 + B * Da * (1-x1) * np.exp(x2) - beta * x2


In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111)
ax.semilogy(B_arr, L2_norm_arr_B, '-')

In [ ]:

unstable_ss, unstable_Da, stable_ss1, stable_Da1, stable_ss2, stable_Da2, Da_arr, x1min, x2min, x1max, x2max, x1LC, x2LC = \
        URP_metrics.make_Bifurc_true_with_LC()


T = 1
teval = np.linspace(0, T, 100)

B = 11
beta = 3

beta_arr = np.sort(
            np.unique(
            np.concatenate((
                np.linspace(1,5,100),
                np.array([3])
            ))
            )
        )

L2_norm_arr_beta = np.zeros(beta_arr.size)

for j in range(beta_arr.size):
    beta = beta_arr[j]

    Da_sampler = default_rng(seed=123)
    LC_sampler = default_rng(seed=456)

    Da_sample = np.zeros(100)
    x1_sample = np.zeros(100)
    x2_sample = np.zeros(100)


    RHS_hair_error_trueLC_True = []
    RHS_hair_error_trueLC_Pred = []

    for i in range(x1_sample.size):
        Da_index = Da_sampler.choice(len(Da_arr))
        LC_index = LC_sampler.choice(np.array(x1LC[Da_index]).size)

        init = np.array([np.array(x1LC[Da_index]).reshape((-1))[LC_index],\
                            np.array(x2LC[Da_index]).reshape((-1))[LC_index]])
        pars_true = get_pars(Da_arr[Da_index], beta=beta, B=B)
        pars_ANN = np.array([Da_arr[Da_index]]).reshape((-1))

        sol_true = solve_ivp(f_cstr, y0=init, t_span=[0, T],
                                t_eval=teval, args=pars_true,
                                rtol=1e-5, atol=1e-8)

        sol_pred = solve_ivp(my_ode, y0=init, t_span=[0, T],
                                t_eval=teval, args=(pars_ANN, B, beta),
                                rtol=1e-5, atol=1e-8)

        RHS_hair_error_trueLC_True.append(sol_true.y[:,-1])
        RHS_hair_error_trueLC_Pred.append(sol_pred.y[:,-1])

    RHS_hair_error_trueLC_Pred = np.array(RHS_hair_error_trueLC_Pred)
    RHS_hair_error_trueLC_True = np.array(RHS_hair_error_trueLC_True)

    scaler = MinMaxScaler()
    scaler.fit(RHS_hair_error_trueLC_True)

    RHS_hair_error_trueLC_Pred_norm = scaler.transform(RHS_hair_error_trueLC_Pred)
    RHS_hair_error_trueLC_True_norm = scaler.transform(RHS_hair_error_trueLC_True)

    L2_norm = np.sum(np.sqrt(np.sum(((RHS_hair_error_trueLC_Pred_norm - RHS_hair_error_trueLC_True_norm)**2), axis=1)), axis=0) / RHS_hair_error_trueLC_True_norm.size

    L2_norm_arr_beta[j] = L2_norm

    if j % 10 == 0:
        print(j, "beta = ", beta)

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111)
ax.semilogy(beta_arr, L2_norm_arr_beta, '-')

In [ ]:
fig = plt.figure(figsize=(8, 3))
ax = fig.add_subplot(121)
ax.semilogy(B_arr, L2_norm_arr_B, '-', linewidth=2, color='green')
ax.set_xlabel('B', fontsize=20)
ax.set_ylabel(r'$\mathcal{L}_2$', fontsize=20)
ax.set_ylim(min(L2_norm_arr_B)*0.9, max(L2_norm_arr_B)*1.1)
ax.vlines(11, min(L2_norm_arr_B)*0.9, max(L2_norm_arr_B)*1.1, linestyle='--', color='red')
ax.set_xticks([10, 11, 12, 13, 14, 15, 16])
ax.tick_params(axis='both', which='major', labelsize=15, direction='in', length=10, width=2)
ax.tick_params(axis='both', which='minor', labelsize=15, direction='in', length=5, width=1)

ax = fig.add_subplot(122)
ax.semilogy(beta_arr, L2_norm_arr_beta, '-', linewidth=2, color='green')
ax.set_xlabel(r'$\beta$', fontsize=20)
ax.set_ylabel(r'$\mathcal{L}_2$', fontsize=20)
ax.set_ylim(min(L2_norm_arr_beta)*0.9, max(L2_norm_arr_beta)*1.1)
ax.vlines(3, min(L2_norm_arr_beta)*0.9, max(L2_norm_arr_beta)*1.1, linestyle='--', color='red')
ax.set_xticks([1, 2, 3, 4, 5])
ax.tick_params(axis='both', which='major', labelsize=15, direction='in', length=10, width=2)
ax.tick_params(axis='both', which='minor', labelsize=15, direction='in', length=5, width=1)

plt.tight_layout()

In [ ]:
RHS_hair_error_trueLC_Pred_norm.shape